In [ ]:
import os
from collections import defaultdict

import gensim.downloader as api
from gensim.models import Word2Vec
import networkx as nx 
from tqdm import tqdm
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import re

## Evaluation

In [ ]:
# Read in a graph (just used to get word lists for analogies)
graph_weighted = nx.read_gexf('data/graph_weighted_with_probs.gexf')
# graph_unweighted = nx.read_gexf('data/pos_tagged_graph_original.gexf')

In [ ]:
pq_to_display = {
    'weighted_graph': [(1, 0.25), (1,0.5), (1,1), (1,2)],
    'unweighted_graph': [(1, 0.25), (1,0.5), (1,1), (1,2)]
}

windows_to_display = {
    'weighted_graph': [3,5,10],
    'unweighted_graph': [3,5,10]
}

INCLUDE_GOOGLE = True      # Mark True if you want to run Analogy Tests and 
                            # Analysis on Google's PreTrained Embeddings

In [ ]:
models = {}         # map of keys to actual models

# lists of model keys
graph_models = []
unweighted_graph_models = []
weighted_graph_models = []
google_models = []
all_models = []

# get google pretrained embeddings
google_model= api.load('word2vec-google-news-300')      # Still needed for some parts, 
                                                        # even if INCLUDE_GOOGLE is False.
                                                        # Used to make tests with a consistent vocab

if INCLUDE_GOOGLE:
    models['google_word2vec'] = google_model
    google_models.append('google_word2vec')

# load unweighted graph embeddings
for (p,q) in pq_to_display['unweighted_graph']:
    for k in windows_to_display['unweighted_graph']:
        graph_key = f'unweighted_p{p}_q{q}_window_{k}'
        models[graph_key] = Word2Vec.load(f"data/unweighted_graph_embeddings_p{p}_q{q}_window{k}.model").wv
        graph_models.append(graph_key)
        unweighted_graph_models.append(graph_key)

# load weighted graph embeddings
for (p,q) in pq_to_display['weighted_graph']:
    for k in windows_to_display['weighted_graph']:
        graph_key = f'weighted_p{p}_q{q}_window_{k}'
        models[graph_key] = Word2Vec.load(f"data/weighted_graph_embeddings_p{p}_q{q}_window{k}.model").wv
        graph_models.append(graph_key)
        weighted_graph_models.append(graph_key)

all_models = google_models + graph_models

In [ ]:
# Create an index mapping words to nodes that match
word_node_index = defaultdict(list)
for node, data in graph_weighted.nodes(data=True):
    if data['type'] != 'meta':
        word = data['value']
        word_node_index[word].append((node, data))

In [ ]:
def get_nodes(word, *pos_info):
    nodes = [n for n, _ in word_node_index[word.lower()]]
    if len(pos_info) == 0:
        return nodes

    valid_nodes = []
    for node in nodes:
        match = True
        for item in pos_info:
            if f"'{item}'" not in node:
                match=False
                break
        if match:
            valid_nodes.append(node)

    return valid_nodes

#### Analogy Tests

### Prep

In [ ]:
# read in analogies
analogies_folder = 'analogies'
analogies_master = {}
for analogy_file in os.listdir(analogies_folder):
    category = analogy_file[:-4]
    analogies_master[category] = pd.read_table(os.path.join(analogies_folder, analogy_file), sep=" ", header=None)
    analogies_master[category].columns = ['w1', 'r1', 'w2', 'r2']
    

In [ ]:
# get and save a list of all nodes for words in the analogies
words_not_in_graph = []
for category, tests in analogies_master.items():
    for ind, row in tqdm(tests.iterrows(), category, total=len(tests)):
        for word in row:
            if word not in word_node_index:
                if len(get_nodes(word)) == 0:
                    words_not_in_graph.append(word)

print(f'Words Not in Graph: {words_not_in_graph}')

In [ ]:
# clean analogies
for category, tests in analogies_master.items():
    analogies_master[category] = tests[~tests.isin(words_not_in_graph).any(axis=1)]

### Tests

In [ ]:
google_words = set([word.lower() for word in google_model.index_to_key])

In [ ]:
# create lists of analogies, but in a format of the embeddings models can take. 
# for graphical models, this generates analogies using nodes.
# for google, this generates analogies using words themselves.
# filters out analogies that have words that are either not in the graph or not in google's list
analogy_nodes = defaultdict(list)
analogy_google = defaultdict(list)
for category, tests in analogies_master.items():
    for ind, row in tqdm(tests.iterrows(), category, total=len(tests)):
        w1, r1, w2, r2 = row
        w1_nodes = get_nodes(w1)
        r1_nodes = get_nodes(r1)
        w2_nodes = get_nodes(w2)
        r2_nodes = get_nodes(r2)

        nodes1_3 = set()
        for node1 in w1_nodes:     # for each node that is the first word in the analogy
            try:
                node3 = [node for node in w2_nodes if eval(node)[1:] == eval(node1)[1:]][0]    # find the node for the third word that has the same pos
                nodes1_3.add((node1, node3))
            except IndexError:
                continue
            
            
        nodes2_4 = set() 
        for node2 in r1_nodes:    # then for each word that is the second word in the analogy
            try:
                node4 = [node for node in r2_nodes if eval(node)[1:] == eval(node2)[1:]][0]    # find the node for the third word that has the same pos
                nodes2_4.add((node2, node4))
            except IndexError:
                continue

        for node1, node3 in nodes1_3:
            for node2, node4 in nodes2_4:
                if (eval(node1)[0] in google_words) and (eval(node2)[0] in google_words) \
                        and (eval(node3)[0] in google_words) and (eval(node4)[0] in google_words):
                    
                    analogy_nodes[category].append((node1, node2, node3, node4))
                    analogy_google[category].append((w1, r1, w2, r2))
                    
                    
            

In [ ]:

# function to calculate accuracies using analogies for the different models
def calculate_analogy_accuracy(analogies_list, model_keys, accuracies):
    for category, analogies in analogies_list.items():
        for analogy in tqdm(analogies, f'calculating for {category}...'):
            w1, r1, w2, r2 = analogy
            
            for model_key in model_keys:
                model = models[model_key]
                
                try:
                    r2_hat = model[w2] + (model[r1] - model[w1])
                except KeyError:
                    print(model_key, w1, r1, w2, r2)
                    continue
                most_similar = [item[0] for item in model.similar_by_vector(r2_hat, topn=10)]
    
                if r2 in most_similar[:1]:
                    accuracies[model_key][category]['top1'] += 1
                    accuracies[model_key][category]['top5'] += 1
                    accuracies[model_key][category]['top10'] += 1
                elif r2 in most_similar[:5]:
                    accuracies[model_key][category]['top5'] += 1
                    accuracies[model_key][category]['top10'] += 1
                elif r2 in most_similar[:10]:
                    accuracies[model_key][category]['top10'] += 1


In [ ]:
# initialize
accuracies = {}
for model in models:
    accuracies[model] = {}
    for category in analogy_nodes:
        accuracies[model][category] = {}
        accuracies[model][category]['top1'] = 0
        accuracies[model][category]['top5'] = 0
        accuracies[model][category]['top10'] = 0

# calculate accuracies for the four different models
print('Graph models:')
calculate_analogy_accuracy(analogy_nodes, graph_models, accuracies)

if INCLUDE_GOOGLE:
    print('Google model:')
    calculate_analogy_accuracy(analogy_google, google_models, accuracies)

# normalize accuracies
for model in models:
    for category in analogy_nodes:
        num = len(analogy_nodes[category])
        for count_key in accuracies[model][category]:
            accuracies[model][category][count_key] /= num     

### Display Results

In [ ]:
def display_results(model_list: list[str], p='', q=''):
    _model_list = [mk for mk in model_list if f'p{p}_q{q}_' in mk]
    for model_key in _model_list:
        print(model_key)
        display(pd.DataFrame(accuracies[model_key]))

#### Display Unweighted Graph Embedding Results

##### p=1, q=0.25

In [ ]:
display_results(unweighted_graph_models, p=1,q=0.25)

##### p=1, q=0.5

In [ ]:
display_results(unweighted_graph_models, p=1,q=0.5)

##### p=1, q=1

In [ ]:
display_results(unweighted_graph_models, p=1,q=1)

##### p=1, q=2

In [ ]:
display_results(unweighted_graph_models, p=1,q=2)

#### Display Weighted Graph Embedding Results

##### p=1, q=0.25

In [ ]:
display_results(weighted_graph_models, p=1, q=0.25)

##### p=1, q=0.5

In [ ]:
display_results(weighted_graph_models, p=1, q=0.5)

##### p=1, q=1

In [ ]:
display_results(weighted_graph_models, p=1, q=1)

##### p=1, q=2

In [ ]:
display_results(weighted_graph_models, p=1, q=2)

#### Display Google Embedding Results

In [ ]:
if INCLUDE_GOOGLE:
    display_results(google_models)

#### Display Custom List of Results

In [ ]:
custom_list = []        # fill in here with model_keys, e.g. weighted_graph_p1_q1_window3
display_results(custom_list)

### Bias Check

#### Define Terms to Check
**NOTE:** This is a rough list generated from ChatGPT and may not reflect all opinions. 

In [ ]:
# Occupations

men_occupations = [
    'Engineer',
    'Construction',
    'Firefighter',
    'Mechanic',
    'Plumber',
    'Electrician',
    'Pilot',
    'Information_Technology',
    'Soldier',
    'CEO',
    'Computer_Programmer'
]

women_occupations = [
    'Nurse',
    'Teacher',
    'Secretary',
    'Receptionist',
    'Nanny',
    'Librarian',
    'Waitress',
    'Hairdresser',
    'Flight_attendant',
    'Dancer',
    'Homemaker'
]

In [ ]:
# Sports

sports_men = [
    'Football',
    'Soccer',
    'Baseball',
    'Basketball',
    'Boxing',
    'Rugby',
    'Wrestling',
    'Ice_Hockey',
    'Golf',
]

sports_women = [
    'Gymnastics',
    'Figure_Skating',
    'Swimming',
    'Ballet',
    'Cheerleading',
    'Equestrian',
    'Netball',
    'Volleyball'
]

In [ ]:
# Instruments

instruments_men = [
    'Electric_Guitar',
    'Drums',
    'Trumpet',
    'Saxophone',
    'Bass_Guitar',
    'Trombone',
    'Violin',
    'Bass',
    'Cello',
]

instruments_women = [
    'Flute',
    'Harp',
    'Clarinet',
    'Viola',
    'Piano',
    'Oboe',
    'Harp',
    'Keyboard',
]

In [ ]:
mens = men_occupations + sports_men + instruments_men
womens = women_occupations + sports_women + instruments_women

#### Make Plots

In [ ]:
# Define plot titles:
plot_titles = {}

plot_titles['google_word2vec'] = 'Google Word2Vec Model'

# create plot titles for graph models
for graph_key in graph_models:
    match = re.search(r'(\d+)$', graph_key)
    window_size = match.group(1) if match else ''
    plot_titles[graph_key] = f"{'Unw' if 'unweighted' in graph_key else 'W'}eighted Graph - Node2Vec Window Size {window_size}"

In [ ]:
def calculate_plot_points(model_key):
    model = models[model_key]
    if model_key != 'google_word2vec':  
        man_key = "('man', 'n', 'singular', 'common')"
        woman_key = "('woman', 'n', 'singular', 'common')"
    else:
        man_key = 'man'
        woman_key = 'woman'
        
    man_emb = model[man_key]
    woman_emb = model[woman_key]
    magnitude_man = np.linalg.norm(man_emb)
    magnitude_woman = np.linalg.norm(woman_emb)

    mens_x = []
    mens_y = []

    womens_x = []
    womens_y = []
    
    for item in mens + womens:
        if model_key != 'google_word2vec':  
            nodes = [node for node in get_nodes(item) if 'UNK_POS' not in node]
        else:
            nodes = [item]

        for node in nodes:
            item_emb = model[node]
            magnitude_item = np.linalg.norm(item_emb)
            
            similarity_man = np.dot(item_emb, man_emb) / (magnitude_man * magnitude_item)
            similarity_woman = np.dot(item_emb, woman_emb) / (magnitude_woman * magnitude_item)
            #ax.annotate(node, (similarity_man, similarity_woman), textcoords="offset points", xytext=(5, 5), ha='left', fontsize=8)
            if item in mens:
                mens_x.append(similarity_man)
                mens_y.append(similarity_woman)
            else:
                womens_x.append(similarity_man)
                womens_y.append(similarity_woman)
                
    return mens_x, mens_y, womens_x, womens_y
    

In [ ]:
def make_plots_grid(model_list: list[str], grid_width=2, plot_titles=plot_titles, file_name=None):
    num_plots = len(model_list)
    grid_height = num_plots // grid_width + (0 if num_plots % grid_width == 0 else 1)
    fig, axs = plt.subplots(grid_height, grid_width, figsize=(18, 18))
    i, j = (0, 0)

    for model_key in tqdm(model_list):
        mens_x, mens_y, womens_x, womens_y = calculate_plot_points(model_key)

        ax = axs[i,j] if grid_height > 1 else axs[j]
        ax.scatter(mens_x, mens_y, color='blue', label='Labeled Men\'s')
        ax.scatter(womens_x, womens_y, color='orange', label='Labeled Women\'s')
        ax.legend()
        ax.set_xlabel('Similarity to Man')
        ax.set_ylabel('Similarity to Woman')
        ax.set_title(plot_titles[model_key])
        ax.set_aspect('equal', adjustable='box')
        j = (j + 1) % grid_width
        if j == 0:
            i += 1


    while (j % grid_width) != 0:
        subplot = axs[i,j] if grid_height > 1 else axs[j]
        subplot.set_visible(False)
        j += 1
    
    if file_name:
        plt.savefig(f'../figures/{file_name}.png')
        print(f'plots saved to ../figures/{file_name}.png')

    plt.show()

In [ ]:
def display_and_save_bias_plots(model_list: list[str], p='', q=''):
    _model_list = [mk for mk in model_list if f'p{p}_q{q}_' in mk]
    batches = []
    k = 0
    while k < len(_model_list):
        j = k
        k += 4
        batches.append(_model_list[j:k])

    for i, batch in enumerate(batches):
        print(batch)
        group_label = f'p{p}_' if p != '' else ''
        group_label += f'q{q}_' if q != '' else ''
        make_plots_grid(model_list=batch, file_name=f'bias_plots_{group_label}batch{i}')

##### Google Embeddings

In [ ]:
if INCLUDE_GOOGLE:
    display_and_save_bias_plots(model_list=google_models)

##### Unweighted Graph Embeddings

In [ ]:
display_and_save_bias_plots(model_list=unweighted_graph_models, p=1, q=0.25)

In [ ]:
display_and_save_bias_plots(model_list=unweighted_graph_models, p=1, q=0.5)

In [ ]:
display_and_save_bias_plots(model_list=unweighted_graph_models, p=1, q=1)

In [ ]:
display_and_save_bias_plots(model_list=unweighted_graph_models, p=1, q=2)

##### Weighted Graph Embeddings

In [ ]:
display_and_save_bias_plots(model_list=weighted_graph_models, p=1, q=0.25)

In [ ]:
display_and_save_bias_plots(model_list=weighted_graph_models, p=1, q=0.5)

In [ ]:
display_and_save_bias_plots(model_list=weighted_graph_models, p=1, q=1)

In [ ]:
display_and_save_bias_plots(model_list=weighted_graph_models, p=1, q=2)